# Lap Time Model

Goal: predict lap time from tyre state, fuel load and circuit, then use the model to find the optimal pit-stop lap.

In [154]:
import pandas as pd
import numpy as np

laps = pd.read_parquet("../data/processed/laps_model.parquet")
print(laps.shape)

(78233, 37)


In [155]:
race_driver = ["Year", "Round", "Driver"]

laps["DriverRaceMedian"] = laps.groupby(race_driver)["LapTimeSec"].transform("median")
laps["LapDelta"] = laps["LapTimeSec"] - laps["DriverRaceMedian"]

laps["LapDelta"].describe()

count    78233.000000
mean         0.027794
std          1.122637
min         -8.463000
25%         -0.621500
50%          0.000000
75%          0.650000
max         37.930000
Name: LapDelta, dtype: float64

In [156]:
features = ["TyreLife", "Compound", "LapNumber", "Event", "Year", "Team", "FreshTyre"]
target = "LapDelta"

model_data = laps[features + [target]].copy()

for col in ["Compound", "Event", "Team"]:
    model_data[col] = model_data[col].astype("category")

model_data.dtypes

TyreLife      float64
Compound     category
LapNumber     float64
Event        category
Year            int64
Team         category
FreshTyre        bool
LapDelta      float64
dtype: object

In [157]:
is_train = model_data["Year"] <= 2024
is_test = model_data["Year"] == 2025

X_train, y_train = model_data.loc[is_train, features], model_data.loc[is_train, target]
X_test, y_test = model_data.loc[is_test, features], model_data.loc[is_test, target]

print("train:", X_train.shape, " test:", X_test.shape)

train: (57352, 7)  test: (20881, 7)


In [158]:
from sklearn.metrics import mean_absolute_error

naive_pred = np.zeros(len(y_test))
naive_mae = mean_absolute_error(y_test, naive_pred)
print(f"Naive MAE: {naive_mae:.3f} s")

Naive MAE: 0.796 s


In [159]:
from sklearn.linear_model import LinearRegression

X_all_lin = pd.get_dummies(
    model_data[features],
    columns=["Compound", "Event", "Team"],
    drop_first=True,
    dtype=float,
)
X_train_lin = X_all_lin[is_train]
X_test_lin = X_all_lin[is_test]

lin = LinearRegression()
lin.fit(X_train_lin, y_train)

lin_pred = lin.predict(X_test_lin)
lin_mae = mean_absolute_error(y_test, lin_pred)
print(f"Linear MAE: {lin_mae:.3f} s")

Linear MAE: 0.594 s


In [160]:
from lightgbm import LGBMRegressor

X_train_gb = X_train.astype({"FreshTyre": int})
X_test_gb = X_test.astype({"FreshTyre": int})

gbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)
gbm.fit(X_train_gb, y_train)

gbm_pred = gbm.predict(X_test_gb)
gbm_mae = mean_absolute_error(y_test, gbm_pred)
print(f"LightGBM MAE: {gbm_mae:.3f} s")

LightGBM MAE: 0.603 s


In [161]:
results = pd.DataFrame({
    "Model": ["Naive (predict 0)", "Linear regression", "LightGBM"],
    "MAE (s)": [naive_mae, lin_mae, gbm_mae],
})
results["Improvement vs naive"] = (1 - results["MAE (s)"] / naive_mae) * 100
results.round(3)

,Model,MAE (s),Improvement vs naive
0,Naive (predict 0),0.796,0.000
1,Linear regression,0.594,25.365
2,LightGBM,0.603,24.210


In [162]:
lin_train_mae = mean_absolute_error(y_train, lin.predict(X_train_lin))
gbm_train_mae = mean_absolute_error(y_train, gbm.predict(X_train_gb))

print(f"Linear   train: {lin_train_mae:.3f}   test: {lin_mae:.3f}")
print(f"LightGBM train: {gbm_train_mae:.3f}   test: {gbm_mae:.3f}")

Linear   train: 0.587   test: 0.594
LightGBM train: 0.396   test: 0.603


## 5.10 Model parameters

In [163]:
print(gbm.get_params())

coef = pd.Series(lin.coef_, index=X_train_lin.columns)
coef[["TyreLife", "LapNumber", "Compound_MEDIUM", "Compound_SOFT", "FreshTyre"]].round(4)

{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': None, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': -1}


TyreLife           0.0274
LapNumber         -0.0468
Compound_MEDIUM    0.0758
Compound_SOFT     -0.0006
FreshTyre          0.0631
dtype: float64

In [164]:
is_tr = model_data["Year"] <= 2023
is_val = model_data["Year"] == 2024

X_tr = model_data.loc[is_tr, features].astype({"FreshTyre": int})
y_tr = model_data.loc[is_tr, target]
X_val = model_data.loc[is_val, features].astype({"FreshTyre": int})
y_val = model_data.loc[is_val, target]

print("train:", X_tr.shape, " val:", X_val.shape, " test:", X_test_gb.shape)

train: (36359, 7)  val: (20993, 7)  test: (20881, 7)


In [165]:
import lightgbm as lgb

gbm_reg = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=200,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)
gbm_reg.fit(
    X_tr, y_tr,
    eval_X=X_val,
    eval_y=y_val,
    eval_metric="l1",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)

best_iter = gbm_reg.best_iteration_
print("best number of trees:", best_iter)

best number of trees: 77


In [166]:
final_gbm = LGBMRegressor(**{**gbm_reg.get_params(), "n_estimators": best_iter})
final_gbm.fit(X_train_gb, y_train)

final_train_mae = mean_absolute_error(y_train, final_gbm.predict(X_train_gb))
final_test_mae = mean_absolute_error(y_test, final_gbm.predict(X_test_gb))
print(f"Regularised LightGBM  train: {final_train_mae:.3f}   test: {final_test_mae:.3f}")

Regularised LightGBM  train: 0.526   test: 0.554


## 5.15 Experiment tracking with MLflow

In [167]:
import mlflow
from pathlib import Path

db_path = Path("../mlflow.db").resolve().as_posix()
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("lap-time-model")

<Experiment: artifact_location=('file:///c:/Users/vova1/Documents/vscodefiles/resume projects/f1 '
 'project/notebooks/mlruns/1'), creation_time=1790736436955, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790736436955, lifecycle_stage='active', name='lap-time-model', tags={}, trace_location=None, workspace='default'>

In [168]:
def log_run(name, params, train_mae, test_mae):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        mlflow.log_metric("train_mae", train_mae)
        mlflow.log_metric("test_mae", test_mae)

In [169]:
log_run("linear_regression", {"model": "LinearRegression"}, lin_train_mae, lin_mae)

log_run("lightgbm_default",
        {"model": "LightGBM", "n_estimators": 500, "num_leaves": 31, "min_child_samples": 20},
        gbm_train_mae, gbm_mae)

log_run("lightgbm_regularised",
        {"model": "LightGBM", "n_estimators": best_iter, "num_leaves": 15,
         "min_child_samples": 200, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0},
        final_train_mae, final_test_mae)

## 5.16 Save and register the final model

In [170]:
import mlflow.lightgbm

with mlflow.start_run(run_name="lightgbm_final"):
    mlflow.log_params(final_gbm.get_params())
    mlflow.log_metric("train_mae", final_train_mae)
    mlflow.log_metric("test_mae", final_test_mae)
    mlflow.lightgbm.log_model(
        final_gbm,
        name="model",
        input_example=X_test_gb.head(5),
        registered_model_name="lap-time-lgbm",
    )

c:\Users\vova1\miniconda3\envs\f1\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/04 15:12:26 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: MlflowException("Failed to enforce schema of data '       TyreLife Compound  LapNumber                  Event  Year

## Q5. On which lap should a driver pit to minimise total race time?

# Pit-Stop Strategy

## 5A. Pit-stop time loss per circuit

### 5A.1 Setup

In [171]:
import fastf1
import pandas as pd
import numpy as np

fastf1.Cache.enable_cache("../cache")
fastf1.set_log_level("ERROR")

laps_model = pd.read_parquet("../data/processed/laps_model.parquet")

### 5A.2 Pit loss for one race

In [172]:
def pit_loss_for_race(year, rnd):
    session = fastf1.get_session(year, rnd, "R")
    session.load(telemetry=False, weather=False, messages=False)
    laps = session.laps.copy()
    laps["Sec"] = laps["LapTime"].dt.total_seconds()

    normal = laps[laps["PitInTime"].isna() & laps["PitOutTime"].isna() & (laps["TrackStatus"] == "1")]
    typical = normal.groupby("Driver")["Sec"].median()

    rows = []
    for _, in_lap in laps[laps["PitInTime"].notna()].iterrows():
        driver = in_lap["Driver"]
        out_lap = laps[(laps["Driver"] == driver) & (laps["LapNumber"] == in_lap["LapNumber"] + 1)]
        if out_lap.empty or driver not in typical:
            continue
        out_lap = out_lap.iloc[0]
        if in_lap["TrackStatus"] != "1" or out_lap["TrackStatus"] != "1":
            continue
        loss = in_lap["Sec"] + out_lap["Sec"] - 2 * typical[driver]
        rows.append({
            "Year": year, "Round": rnd, "Driver": driver, "PitLoss": loss,
            "LapNumber": int(in_lap["LapNumber"]),
            "From": in_lap["Compound"], "To": out_lap["Compound"],
        })

    return pd.DataFrame(rows)

In [173]:
monza = pit_loss_for_race(2024, 16)
print(monza["PitLoss"].describe())
monza.head()

count    30.000000
mean     27.532367
std       4.318844
min      22.648000
25%      24.364500
50%      26.552500
75%      28.423250
max      41.391000
Name: PitLoss, dtype: float64


,Year,Round,Driver,PitLoss,LapNumber,From,To
0,2024,16,LEC,27.644,15,MEDIUM,HARD
1,2024,16,PIA,27.229,16,MEDIUM,HARD
2,2024,16,PIA,24.603,38,HARD,HARD
3,2024,16,NOR,26.055,14,MEDIUM,HARD
4,2024,16,NOR,24.285,32,HARD,HARD


In [174]:
races = (
    laps_model[laps_model["Year"] >= 2024][["Year", "Round", "Event"]]
    .drop_duplicates()
    .sort_values(["Year", "Round"])
)

all_stops = []
for _, race in races.iterrows():
    try:
        stops = pit_loss_for_race(race["Year"], race["Round"])
        stops["Event"] = race["Event"]
        all_stops.append(stops)
        print(f"{race['Year']} R{race['Round']}: {len(stops)} stops")
    except Exception as e:
        print(f"{race['Year']} R{race['Round']}: FAILED - {e}")

pit_stops = pd.concat(all_stops, ignore_index=True)
print(pit_stops.shape)

2024 R1: 38 stops
2024 R2: 5 stops
2024 R3: 31 stops
2024 R4: 35 stops
2024 R5: 22 stops
2024 R6: 12 stops
2024 R7: 27 stops
2024 R8: 5 stops
2024 R9: 19 stops
2024 R10: 42 stops
2024 R11: 44 stops
2024 R12: 45 stops
2024 R13: 40 stops
2024 R14: 32 stops
2024 R15: 26 stops
2024 R16: 30 stops
2024 R17: 17 stops
2024 R18: 23 stops
2024 R19: 21 stops
2024 R20: 21 stops
2024 R22: 39 stops
2024 R23: 8 stops
2024 R24: 22 stops
2025 R1: 0 stops
2025 R2: 24 stops
2025 R3: 21 stops
2025 R4: 28 stops
2025 R5: 16 stops
2025 R6: 9 stops
2025 R7: 12 stops
2025 R8: 36 stops
2025 R9: 40 stops
2025 R10: 28 stops
2025 R11: 30 stops
2025 R12: 16 stops
2025 R13: 26 stops
2025 R14: 29 stops
2025 R15: 10 stops
2025 R16: 19 stops
2025 R17: 15 stops
2025 R18: 22 stops
2025 R19: 18 stops
2025 R20: 26 stops
2025 R21: 33 stops
2025 R22: 16 stops


Request for URL https://api.jolpi.ca/ergast/f1/2025/23/laps/1.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\vova1\miniconda3\envs\f1\Lib\site-packages\requests_cache\session.py", line 316, in _resend
    response.raise_for_status()
  File "c:\Users\vova1\miniconda3\envs\f1\Lib\site-packages\requests\models.py", line 1167, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2025/23/laps/1.json


2025 R23: 23 stops


Request for URL https://api.jolpi.ca/ergast/f1/2025/24/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\vova1\miniconda3\envs\f1\Lib\site-packages\requests_cache\session.py", line 316, in _resend
    response.raise_for_status()
  File "c:\Users\vova1\miniconda3\envs\f1\Lib\site-packages\requests\models.py", line 1167, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2025/24/results.json


2025 R24: 27 stops
(1128, 8)


In [175]:
pit_stops = pit_stops[pit_stops["PitLoss"].between(10, 45)]

pit_loss = (
    pit_stops.groupby("Event")["PitLoss"]
    .agg(["median", "count"])
    .rename(columns={"median": "PitLoss", "count": "Stops"})
    .sort_values("PitLoss")
    .reset_index()
)
pit_loss.round(1)

,Event,PitLoss,Stops
0,Monaco Grand Prix,19.1,41
1,Canadian Grand Prix,19.7,45
2,Saudi Arabian Grand Prix,20.2,21
3,Belgian Grand Prix,20.3,45
4,Australian Grand Prix,20.5,30
5,Hungarian Grand Prix,20.5,69
6,Miami Grand Prix,20.6,21
7,Austrian Grand Prix,21.3,74
8,United States Grand Prix,21.8,39
9,Azerbaijan Grand Prix,21.9,32


In [176]:
pit_loss.to_parquet("../data/processed/pit_loss.parquet")

## 5B. Strategy simulator

In [177]:
import mlflow
from pathlib import Path

db_path = Path("../mlflow.db").resolve().as_posix()
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

model = mlflow.lightgbm.load_model("models:/lap-time-lgbm/1")
print(type(model))

<class 'lightgbm.sklearn.LGBMRegressor'>


In [178]:
print(sorted(laps_model.loc[laps_model["Year"] == 2025, "Team"].unique()))
print(sorted(laps_model.loc[laps_model["Year"] == 2025, "Event"].unique()))

['Alpine', 'Aston Martin', 'Ferrari', 'Haas F1 Team', 'Kick Sauber', 'McLaren', 'Mercedes', 'Racing Bulls', 'Red Bull Racing', 'Williams']
['Abu Dhabi Grand Prix', 'Australian Grand Prix', 'Austrian Grand Prix', 'Azerbaijan Grand Prix', 'Bahrain Grand Prix', 'Belgian Grand Prix', 'British Grand Prix', 'Canadian Grand Prix', 'Chinese Grand Prix', 'Dutch Grand Prix', 'Emilia Romagna Grand Prix', 'Hungarian Grand Prix', 'Italian Grand Prix', 'Japanese Grand Prix', 'Las Vegas Grand Prix', 'Mexico City Grand Prix', 'Miami Grand Prix', 'Monaco Grand Prix', 'Qatar Grand Prix', 'Saudi Arabian Grand Prix', 'Singapore Grand Prix', 'Spanish Grand Prix', 'São Paulo Grand Prix', 'United States Grand Prix']


In [179]:
FEATURES = ["TyreLife", "Compound", "LapNumber", "Event", "Year", "Team", "FreshTyre"]

def simulate_one_stop(event, team, year, first, second, total_laps, pit_loss_sec, min_stint=8):
    results = []
    for pit_lap in range(min_stint, total_laps - min_stint + 1):
        lap_numbers = np.arange(1, total_laps + 1)
        on_first = lap_numbers <= pit_lap

        race = pd.DataFrame({
            "TyreLife": np.where(on_first, lap_numbers, lap_numbers - pit_lap).astype(float),
            "Compound": np.where(on_first, first, second),
            "LapNumber": lap_numbers.astype(float),
            "Event": event,
            "Year": year,
            "Team": team,
            "FreshTyre": 1,
        })
        for col in ["Compound", "Event", "Team"]:
            race[col] = race[col].astype("category")

        total = model.predict(race[FEATURES]).sum() + pit_loss_sec
        results.append({"PitLap": pit_lap, "TotalDelta": total})

    return pd.DataFrame(results)

In [180]:
event = "Italian Grand Prix"
team = "Ferrari"
total_laps = 53
monza_loss = pit_loss.loc[pit_loss["Event"] == event, "PitLoss"].iloc[0]

sim = simulate_one_stop(event, team, 2025, "MEDIUM", "HARD", total_laps, monza_loss)
best = sim.loc[sim["TotalDelta"].idxmin()]
print(f"Best pit lap: {int(best['PitLap'])}")
sim.head()

Best pit lap: 36


,PitLap,TotalDelta
0,8,33.783426
1,9,33.613697
2,10,33.442016
3,11,33.281460
4,12,33.180485


In [181]:
import plotly.express as px

sim["LossVsBest"] = sim["TotalDelta"] - sim["TotalDelta"].min()

fig = px.line(
    sim, x="PitLap", y="LossVsBest", markers=True,
    labels={"PitLap": "Lap of the pit stop", "LossVsBest": "Time lost vs best strategy (s)"},
    title=f"{event}: MEDIUM → HARD, when to pit",
)
fig.add_vline(x=best["PitLap"], line_dash="dash", line_color="green",
              annotation_text=f"best: lap {int(best['PitLap'])}")
fig.show()

In [182]:
race_length = (
    laps_model[laps_model["Year"] == 2025]
    .groupby("Event")["LapNumber"].max()
    .astype(int)
)

windows = []
for event_name, n_laps in race_length.items():
    if event_name not in set(pit_loss["Event"]):
        continue
    loss = pit_loss.loc[pit_loss["Event"] == event_name, "PitLoss"].iloc[0]
    s = simulate_one_stop(event_name, team, 2025, "MEDIUM", "HARD", n_laps, loss)
    s["LossVsBest"] = s["TotalDelta"] - s["TotalDelta"].min()

    in_window = s[s["LossVsBest"] <= 1.0]["PitLap"]
    windows.append({
        "Event": event_name,
        "RaceLaps": n_laps,
        "BestLap": int(s.loc[s["TotalDelta"].idxmin(), "PitLap"]),
        "WindowStart": int(in_window.min()),
        "WindowEnd": int(in_window.max()),
    })

windows = pd.DataFrame(windows).sort_values("BestLap")
windows

,Event,RaceLaps,BestLap,WindowStart,WindowEnd
17,Monaco Grand Prix,78,8,8,10
18,Qatar Grand Prix,57,9,8,14
1,Australian Grand Prix,43,11,8,33
5,Belgian Grand Prix,44,11,8,33
6,British Grand Prix,52,32,28,36
14,Las Vegas Grand Prix,50,33,8,37
13,Japanese Grand Prix,53,33,30,37
3,Azerbaijan Grand Prix,51,33,29,37
19,Saudi Arabian Grand Prix,50,33,29,37
4,Bahrain Grand Prix,57,34,31,39


In [183]:
windows["WindowLength"] = windows["WindowEnd"] - windows["WindowStart"]

fig = px.bar(
    windows, y="Event", x="WindowLength", base="WindowStart", orientation="h",
    labels={"x": "Lap", "Event": ""},
    title="One-stop pit window by circuit (MEDIUM → HARD, within 1 s of the best lap)",
)
fig.add_scatter(x=windows["BestLap"], y=windows["Event"], mode="markers",
                marker=dict(color="red", size=10), name="Best lap")
fig.update_layout(height=750, xaxis_title="Lap of the pit stop")
fig.show()
fig.write_image("../images/pit_windows.png", scale=2, height=750)

In [184]:
real = laps_model[laps_model["Year"] == 2025]

stint_info = (
    real.groupby(["Event", "Driver", "Stint"])
    .agg(Compound=("Compound", "first"), LastLap=("LapNumber", "max"))
    .reset_index()
)
stint_info.head()

,Event,Driver,Stint,Compound,LastLap
0,Abu Dhabi Grand Prix,ALB,1.0,SOFT,7.0
1,Abu Dhabi Grand Prix,ALB,2.0,HARD,32.0
2,Abu Dhabi Grand Prix,ALB,3.0,MEDIUM,58.0
3,Abu Dhabi Grand Prix,ALO,1.0,MEDIUM,15.0
4,Abu Dhabi Grand Prix,ALO,2.0,HARD,58.0


In [185]:
s1 = stint_info[stint_info["Stint"] == 1].set_index(["Event", "Driver"])
s2 = stint_info[stint_info["Stint"] == 2].set_index(["Event", "Driver"])
max_stint = real.groupby(["Event", "Driver"])["Stint"].max().rename("MaxStint")

both = s1.join(s2, lsuffix="_1", rsuffix="_2", how="inner").join(max_stint)

is_one_stop = (
    (both["MaxStint"] == 2)
    & (both["Compound_1"] == "MEDIUM")
    & (both["Compound_2"] == "HARD")
)

real_pit = both[is_one_stop].groupby("Event")["LastLap_1"].median().rename("RealPitLap")
real_pit

Event
Abu Dhabi Grand Prix        15.0
Austrian Grand Prix         31.5
Azerbaijan Grand Prix       20.0
Canadian Grand Prix         17.0
Chinese Grand Prix          13.0
Dutch Grand Prix            21.5
Hungarian Grand Prix        32.0
Italian Grand Prix          27.0
Japanese Grand Prix         22.0
Las Vegas Grand Prix        20.5
Miami Grand Prix            32.0
Saudi Arabian Grand Prix    19.0
Singapore Grand Prix        24.0
Name: RealPitLap, dtype: float64

In [186]:
compare = windows.merge(real_pit, on="Event", how="left")
compare[["Event", "RaceLaps", "BestLap", "WindowStart", "WindowEnd", "RealPitLap"]]

,Event,RaceLaps,BestLap,WindowStart,WindowEnd,RealPitLap
0,Monaco Grand Prix,78,8,8,10,NaN
1,Qatar Grand Prix,57,9,8,14,NaN
2,Australian Grand Prix,43,11,8,33,NaN
3,Belgian Grand Prix,44,11,8,33,NaN
4,British Grand Prix,52,32,28,36,NaN
5,Las Vegas Grand Prix,50,33,8,37,20.5
6,Japanese Grand Prix,53,33,30,37,22.0
7,Azerbaijan Grand Prix,51,33,29,37,20.0
8,Saudi Arabian Grand Prix,50,33,29,37,19.0
9,Bahrain Grand Prix,57,34,31,39,NaN


In [187]:
mono = {"TyreLife": 1, "LapNumber": -1}
constraints = [mono.get(f, 0) for f in features]
print(list(zip(features, constraints)))

gbm_mono = LGBMRegressor(**{
    **gbm_reg.get_params(),
    "n_estimators": 3000,
    "monotone_constraints": constraints,
    "monotone_constraints_method": "advanced",
})
gbm_mono.fit(
    X_tr, y_tr,
    eval_X=X_val,
    eval_y=y_val,
    eval_metric="l1",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
best_iter_mono = gbm_mono.best_iteration_
print("best number of trees:", best_iter_mono)

[('TyreLife', 1), ('Compound', 0), ('LapNumber', -1), ('Event', 0), ('Year', 0), ('Team', 0), ('FreshTyre', 0)]
best number of trees: 75


In [188]:
final_mono = LGBMRegressor(**{**gbm_mono.get_params(), "n_estimators": best_iter_mono})
final_mono.fit(X_train_gb, y_train)

mono_train_mae = mean_absolute_error(y_train, final_mono.predict(X_train_gb))
mono_test_mae = mean_absolute_error(y_test, final_mono.predict(X_test_gb))
print(f"Monotone LightGBM  train: {mono_train_mae:.3f}   test: {mono_test_mae:.3f}")
print(f"Previous (v1)      train: {final_train_mae:.3f}   test: {final_test_mae:.3f}")

Monotone LightGBM  train: 0.535   test: 0.553
Previous (v1)      train: 0.526   test: 0.554


In [189]:
with mlflow.start_run(run_name="lightgbm_monotone"):
    mlflow.log_params(final_mono.get_params())
    mlflow.log_metric("train_mae", mono_train_mae)
    mlflow.log_metric("test_mae", mono_test_mae)
    mlflow.lightgbm.log_model(
        final_mono,
        name="model",
        input_example=X_test_gb.head(5),
        registered_model_name="lap-time-lgbm",
    )

c:\Users\vova1\miniconda3\envs\f1\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/04 15:14:14 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: MlflowException("Failed to enforce schema of data '       TyreLife Compound  LapNumber                  Event  Year

### 5D.4 Pit windows with model v2

In [190]:
model = final_mono

windows_v2 = []
for event_name, n_laps in race_length.items():
    if event_name not in set(pit_loss["Event"]):
        continue
    loss = pit_loss.loc[pit_loss["Event"] == event_name, "PitLoss"].iloc[0]
    s = simulate_one_stop(event_name, team, 2025, "MEDIUM", "HARD", n_laps, loss)
    s["LossVsBest"] = s["TotalDelta"] - s["TotalDelta"].min()

    in_window = s[s["LossVsBest"] <= 1.0]["PitLap"]
    windows_v2.append({
        "Event": event_name,
        "RaceLaps": n_laps,
        "BestLap_v2": int(s.loc[s["TotalDelta"].idxmin(), "PitLap"]),
        "WindowStart_v2": int(in_window.min()),
        "WindowEnd_v2": int(in_window.max()),
    })

windows_v2 = pd.DataFrame(windows_v2)

In [191]:
compare_v2 = (
    windows[["Event", "RaceLaps", "BestLap"]]
    .merge(windows_v2[["Event", "BestLap_v2", "WindowStart_v2", "WindowEnd_v2"]], on="Event")
    .merge(real_pit, on="Event", how="left")
    .rename(columns={"BestLap": "BestLap_v1"})
)
compare_v2

,Event,RaceLaps,BestLap_v1,BestLap_v2,WindowStart_v2,WindowEnd_v2,RealPitLap
0,Monaco Grand Prix,78,8,53,45,60,NaN
1,Qatar Grand Prix,57,9,11,8,16,NaN
2,Australian Grand Prix,43,11,11,8,30,NaN
3,Belgian Grand Prix,44,11,11,8,32,NaN
4,British Grand Prix,52,32,33,29,39,NaN
5,Las Vegas Grand Prix,50,33,11,8,17,20.5
6,Japanese Grand Prix,53,33,33,30,38,22.0
7,Azerbaijan Grand Prix,51,33,34,29,39,20.0
8,Saudi Arabian Grand Prix,50,33,33,8,37,19.0
9,Bahrain Grand Prix,57,34,36,31,42,NaN


### 5E. Physics baseline: analytical optimal pit lap

In [192]:
WEAR_MEDIUM = 0.037
WEAR_HARD = 0.031
PACE_GAP = 0.034

analytic = race_length.rename("RaceLaps").reset_index()
analytic["AnalyticPitLap"] = (
    (WEAR_HARD * analytic["RaceLaps"] + PACE_GAP) / (WEAR_MEDIUM + WEAR_HARD)
).round(1)

check = (
    analytic
    .merge(windows_v2[["Event", "BestLap_v2"]], on="Event", how="left")
    .merge(real_pit, on="Event", how="left")
)
check.dropna(subset=["RealPitLap"])[["Event", "RaceLaps", "AnalyticPitLap", "BestLap_v2", "RealPitLap"]]

,Event,RaceLaps,AnalyticPitLap,BestLap_v2,RealPitLap
0,Abu Dhabi Grand Prix,58,26.9,35,15.0
2,Austrian Grand Prix,70,32.4,50,31.5
3,Azerbaijan Grand Prix,51,23.7,34,20.0
7,Canadian Grand Prix,66,30.6,31,17.0
8,Chinese Grand Prix,56,26.0,35,13.0
9,Dutch Grand Prix,72,33.3,34,21.5
11,Hungarian Grand Prix,70,32.4,37,32.0
12,Italian Grand Prix,53,24.7,34,27.0
13,Japanese Grand Prix,53,24.7,33,22.0
14,Las Vegas Grand Prix,50,23.3,11,20.5


In [193]:
eval_df = check.dropna(subset=["RealPitLap"]).merge(
    windows[["Event", "BestLap"]].rename(columns={"BestLap": "BestLap_v1"}), on="Event"
)

errors = pd.Series({
    "LightGBM v1": (eval_df["BestLap_v1"] - eval_df["RealPitLap"]).abs().mean(),
    "LightGBM v2 (monotone)": (eval_df["BestLap_v2"] - eval_df["RealPitLap"]).abs().mean(),
    "Analytical formula": (eval_df["AnalyticPitLap"] - eval_df["RealPitLap"]).abs().mean(),
}).round(1)
errors

LightGBM v1               17.3
LightGBM v2 (monotone)    12.6
Analytical formula         6.0
dtype: float64

In [194]:
plot_df = eval_df.melt(
    id_vars="Event",
    value_vars=["RealPitLap", "AnalyticPitLap", "BestLap_v2"],
    var_name="Method",
    value_name="PitLap",
)
plot_df["Method"] = plot_df["Method"].map({
    "RealPitLap": "Real (2025)",
    "AnalyticPitLap": "Analytical formula",
    "BestLap_v2": "LightGBM v2",
})

order = eval_df.sort_values("RealPitLap")["Event"].tolist()

fig = px.scatter(
    plot_df, x="PitLap", y="Event", color="Method",
    color_discrete_map={"Real (2025)": "green", "Analytical formula": "royalblue", "LightGBM v2": "orange"},
    category_orders={"Event": order},
    labels={"PitLap": "Lap of the pit stop", "Event": ""},
    title="One-stop MEDIUM → HARD: predicted vs real pit lap (2025)",
)
fig.update_traces(marker=dict(size=12))
fig.update_layout(height=600)
fig.show()
fig.write_image("../images/predicted_vs_real_pit.png", scale=2, height=600)

In [195]:
one_stop_ids = both[is_one_stop].reset_index()[["Event", "Driver"]]

green = pit_stops[
    (pit_stops["Year"] == 2025)
    & (pit_stops["From"] == "MEDIUM")
    & (pit_stops["To"] == "HARD")
].merge(one_stop_ids, on=["Event", "Driver"])

real_green = green.groupby("Event")["LapNumber"].median().rename("RealGreenPitLap")

fair = eval_df.merge(real_green, on="Event", how="left")
fair[["Event", "RealPitLap", "RealGreenPitLap", "AnalyticPitLap"]]

,Event,RealPitLap,RealGreenPitLap,AnalyticPitLap
0,Abu Dhabi Grand Prix,15.0,16.0,26.9
1,Austrian Grand Prix,31.5,32.5,32.4
2,Azerbaijan Grand Prix,20.0,28.0,23.7
3,Canadian Grand Prix,17.0,18.0,30.6
4,Chinese Grand Prix,13.0,14.0,26.0
5,Dutch Grand Prix,21.5,NaN,33.3
6,Hungarian Grand Prix,32.0,33.0,32.4
7,Italian Grand Prix,27.0,28.0,24.7
8,Japanese Grand Prix,22.0,23.0,24.7
9,Las Vegas Grand Prix,20.5,22.0,23.3


In [196]:
fair_eval = fair.dropna(subset=["RealGreenPitLap"]).copy()
fair_eval["Error"] = fair_eval["AnalyticPitLap"] - fair_eval["RealGreenPitLap"]

print(f"Tracks compared:       {len(fair_eval)}")
print(f"Mean absolute error:   {fair_eval['Error'].abs().mean():.1f} laps")
print(f"Median absolute error: {fair_eval['Error'].abs().median():.1f} laps")
print(f"Mean signed error:     {fair_eval['Error'].mean():+.1f} laps (positive = formula pits later)")
print(f"Within 5 laps:         {(fair_eval['Error'].abs() <= 5).sum()} of {len(fair_eval)}")

Tracks compared:       12
Mean absolute error:   4.9 laps
Median absolute error: 3.6 laps
Mean signed error:     +3.6 laps (positive = formula pits later)
Within 5 laps:         8 of 12


#### 5F.1 Wear per circuit and compound (2022–2024 only)

In [197]:
train_laps = laps[laps["Year"] <= 2024].copy()

for c in ["SOFT", "MEDIUM", "HARD"]:
    is_c = (train_laps["Compound"] == c).astype(float)
    train_laps[f"Age_{c}"] = train_laps["TyreLife"] * is_c
    train_laps[f"Is_{c}"] = is_c

wear_feats = ["LapNumber", "Age_SOFT", "Age_MEDIUM", "Age_HARD", "Is_SOFT", "Is_MEDIUM"]
cols = ["LapTimeSec"] + wear_feats

rows = []
for event, g in train_laps.groupby("Event"):
    d = g[cols] - g.groupby(race_driver)[cols].transform("mean")
    coefs, *_ = np.linalg.lstsq(d[wear_feats].to_numpy(), d["LapTimeSec"].to_numpy(), rcond=None)
    c = dict(zip(wear_feats, coefs))
    rows.append({
        "Event": event,
        "WearMedium": c["Age_MEDIUM"],
        "WearHard": c["Age_HARD"],
        "PaceGap": -c["Is_MEDIUM"],
    })

track_wear = pd.DataFrame(rows)
track_wear.round(3)

,Event,WearMedium,WearHard,PaceGap
0,Abu Dhabi Grand Prix,0.095,0.056,0.274
1,Australian Grand Prix,0.066,-0.000,0.640
2,Austrian Grand Prix,0.064,0.061,0.155
3,Azerbaijan Grand Prix,0.058,0.026,0.426
4,Bahrain Grand Prix,0.145,0.120,0.065
5,Belgian Grand Prix,0.140,0.084,0.277
6,British Grand Prix,0.048,0.035,0.159
7,Canadian Grand Prix,-0.013,0.026,-0.465
8,Chinese Grand Prix,0.147,0.113,-0.070
9,Dutch Grand Prix,0.046,0.044,-0.094


#### 5F.2 Optimal pit lap with circuit-specific wear

In [198]:
tw = track_wear.merge(race_length.rename("RaceLaps").reset_index(), on="Event")

valid = (tw["WearMedium"] > 0) & (tw["WearHard"] > 0)
tw["TrackPitLap"] = np.where(
    valid,
    (tw["WearHard"] * tw["RaceLaps"] + tw["PaceGap"]) / (tw["WearMedium"] + tw["WearHard"]),
    (WEAR_HARD * tw["RaceLaps"] + PACE_GAP) / (WEAR_MEDIUM + WEAR_HARD),
)
tw["TrackPitLap"] = tw["TrackPitLap"].clip(lower=8, upper=tw["RaceLaps"] - 8).round(1)
tw["UsedFallback"] = ~valid

tw[["Event", "RaceLaps", "WearMedium", "WearHard", "TrackPitLap", "UsedFallback"]].round(3)

,Event,RaceLaps,WearMedium,WearHard,TrackPitLap,UsedFallback
0,Abu Dhabi Grand Prix,58,0.095,0.056,23.3,False
1,Australian Grand Prix,43,0.066,-0.000,20.1,True
2,Austrian Grand Prix,70,0.064,0.061,35.3,False
3,Azerbaijan Grand Prix,51,0.058,0.026,21.0,False
4,Bahrain Grand Prix,57,0.145,0.120,26.0,False
5,Belgian Grand Prix,44,0.140,0.084,17.7,False
6,British Grand Prix,52,0.048,0.035,23.8,False
7,Canadian Grand Prix,66,-0.013,0.026,30.6,True
8,Chinese Grand Prix,56,0.147,0.113,24.1,False
9,Dutch Grand Prix,72,0.046,0.044,34.5,False


#### 5F.3 Global vs circuit-specific formula

In [199]:
result = fair_eval.merge(tw[["Event", "TrackPitLap", "UsedFallback"]], on="Event")
result["ErrGlobal"] = (result["AnalyticPitLap"] - result["RealGreenPitLap"]).abs()
result["ErrTrack"] = (result["TrackPitLap"] - result["RealGreenPitLap"]).abs()

print(f"Global formula:   mean {result['ErrGlobal'].mean():.1f}, median {result['ErrGlobal'].median():.1f} laps")
print(f"Circuit formula:  mean {result['ErrTrack'].mean():.1f}, median {result['ErrTrack'].median():.1f} laps")
result[["Event", "RealGreenPitLap", "AnalyticPitLap", "TrackPitLap", "ErrGlobal", "ErrTrack", "UsedFallback"]].round(1)

Global formula:   mean 4.9, median 3.6 laps
Circuit formula:  mean 6.6, median 6.0 laps


,Event,RealGreenPitLap,AnalyticPitLap,TrackPitLap,ErrGlobal,ErrTrack,UsedFallback
0,Abu Dhabi Grand Prix,16.0,26.9,23.3,10.9,7.3,False
1,Austrian Grand Prix,32.5,32.4,35.3,0.1,2.8,False
2,Azerbaijan Grand Prix,28.0,23.7,21.0,4.3,7.0,False
3,Canadian Grand Prix,18.0,30.6,30.6,12.6,12.6,True
4,Chinese Grand Prix,14.0,26.0,24.1,12.0,10.1,False
5,Hungarian Grand Prix,33.0,32.4,36.5,0.6,3.5,False
6,Italian Grand Prix,28.0,24.7,28.6,3.3,0.6,False
7,Japanese Grand Prix,23.0,24.7,27.3,1.7,4.3,False
8,Las Vegas Grand Prix,22.0,23.3,18.4,1.3,3.6,False
9,Miami Grand Prix,21.0,26.5,31.4,5.5,10.4,False
